# Building GPT from First Principles - Little Shakespeare Training

**Complete implementation of GPT-style Transformer with detailed explanations**

This notebook demonstrates:
- Character-level tokenization
- Complete GPT architecture (all components)
- Training loop with proper optimization
- Text generation

**Structure**: Each section has explanations in markdown cells above the code cells.


## 1. Imports and Setup

**What we need:**
- `torch`: PyTorch for deep learning
- `torch.nn`: Neural network modules
- `torch.nn.functional`: Functional operations (softmax, etc.)
- `Dataset, DataLoader`: For data handling
- `math`: Mathematical functions (sqrt, exp, etc.)
- `requests`: To download dataset
- `Path`: File path handling


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import requests
from pathlib import Path


## 2. Character-Level Tokenizer

**Why Character-Level?**
- Task requirement: vocab_size ~65 (as per mentor's specification)
- Each character maps to unique integer ID
- Simple, no out-of-vocabulary (OOV) issues
- Model learns spelling, formatting, punctuation

**How it works:**
1. Find all unique characters in text
2. Create mappings: character → ID (encode) and ID → character (decode)
3. Vocabulary size = number of unique characters (~65 for English text)


In [ ]:
class CharTokenizer:
    """Character-level tokenizer for Shakespeare dataset (vocab_size ~65)"""
    
    def __init__(self, text):
        # Get all unique characters and sort them
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        
        # Create mappings: string to int (for encoding)
        self.stoi = {ch: i for i, ch in enumerate(chars)}  # string to int
        
        # Create mappings: int to string (for decoding)
        self.itos = {i: ch for i, ch in enumerate(chars)}  # int to string
        
    def encode(self, text):
        """Convert string to list of integers"""
        return [self.stoi.get(c, 0) for c in text]
    
    def decode(self, ids):
        """Convert list of integers to string"""
        return ''.join([self.itos[i] for i in ids])


## 3. Layer Normalization

**Mathematical Formulation:**
```
For input x ∈ ℝ^(batch × seq_len × d):
μ = (1/d) Σᵢ xᵢ           # Mean per feature dimension
σ² = (1/d) Σᵢ (xᵢ - μ)²  # Variance per feature dimension
x̂ = (x - μ) / √(σ² + ε)  # Normalized (ε prevents division by zero)
y = γ · x̂ + β            # Scale (γ) and shift (β) are learnable
```

**Why LayerNorm?**
- Stabilizes training by normalizing activations
- Reduces sensitivity to initialization
- Works better than BatchNorm for variable-length sequences
- Enables deeper networks (pre-norm architecture)

**Key Components:**
- `eps = 1e-5`: Small constant to prevent division by zero
- `scale (γ)`: Learnable parameter to scale normalized values
- `shift (β)`: Learnable parameter to shift normalized values


In [ ]:
class LayerNorm(nn.Module):
    """Layer Normalization: Normalize across feature dimension (per sample)"""
    
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # Small constant to prevent division by zero
        
        # Learnable parameters: γ (scale) and β (shift)
        self.scale = nn.Parameter(torch.ones(emb_dim))      # γ
        self.shift = nn.Parameter(torch.zeros(emb_dim))     # β
    
    def forward(self, x):
        """Normalize input: x → (x - μ) / σ → γ·x̂ + β"""
        # Compute mean and variance across last dimension (features)
        mean = x.mean(dim=-1, keepdim=True)      # μ
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # σ²
        
        # Normalize: x̂ = (x - μ) / √(σ² + ε)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        
        # Scale and shift: y = γ·x̂ + β
        return self.scale * norm_x + self.shift
